In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from pydantic import BaseModel, Field

load_dotenv(override=True)

In [ ]:
llm = ChatOpenAI(model="gpt-5.4-mini")


In [ ]:
@tool  # decorator to register the function as a tool
def get_top_searches(day: str) -> str:  # type hints
    """Return the top internet search for a given day of the week."""  # doc-string
    fake_prices = {
        "Monday": "Brand new day",
        "Tuesday": "The odyssey",
        "Wednesday": "Kimi K3",
        "Thursday": "Petal",
        "Friday": "Fifa",
        "Saturday": "Food spots",
        "Sunday": "Monday blues",
    }
    return fake_prices.get(day.title(), "Not a day of the week")

In [ ]:
llm_with_tools = llm.bind_tools(
    [get_top_searches]
)  # we get an LLM object with tools equipped, which we can then invoke on it.

response = llm_with_tools.invoke("What is the top search for today which is thursday?")
print("content:", repr(response.content))
print("tool_calls:", response.tool_calls)


In [ ]:
# Start the conversation and keep the model's tool request in the history
conversation = [HumanMessage("What is the top search for today which is thursday?")]
ai_message = llm_with_tools.invoke(conversation)
conversation.append(ai_message)

# Run each requested tool and add its result as a ToolMessage
for call in ai_message.tool_calls:
    print(call)
    if call["name"] == "get_top_searches":
        result = get_top_searches.invoke(call["args"])
        conversation.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

# Invoke again, now that the model can see the tool result
final = llm_with_tools.invoke(conversation)
print(final.content)
